# 03 · Modelling
Run once with `target = 'street_rate'`, then again with `'comparison_rate'`.

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import shap

In [ ]:
study = gpd.read_file('../data/processed/study_area.gpkg')

target = 'street_rate'          # change to 'comparison_rate' for the second run

## 1. Inputs and split
Population in thousands; crime rates log-transformed with `np.log1p()`.

In [ ]:
study['population_1000'] = study['population'] / 1000

features = ['slai', 'population_1000']
if 'n_poi' in study.columns:
    features.append('n_poi')

X = study[features]
y = np.log1p(study[target])
features

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print('Training squares:', len(X_train))
print('Test squares:', len(X_test))

## 2. Linear regression

In [ ]:
from sklearn.linear_model import LinearRegression

linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

In [ ]:
coefficients = pd.Series(linear_model.coef_, index=features)
coefficients

In [ ]:
from sklearn.metrics import root_mean_squared_error, r2_score

linear_test_predictions = linear_model.predict(X_test)
print('Linear RMSE:', root_mean_squared_error(y_test, linear_test_predictions))
print('Linear R²:', r2_score(y_test, linear_test_predictions))

SHAP explanations.

In [ ]:
background = X_train.sample(100, random_state=42)

explainer = shap.LinearExplainer(linear_model, background)
shap_values = explainer(X_test)

In [ ]:
shap.plots.waterfall(shap_values[0])

In [ ]:
shap.plots.beeswarm(shap_values)

## 3. Decision tree and random forest

In [ ]:
from sklearn.tree import DecisionTreeRegressor, plot_tree

tree_model = DecisionTreeRegressor(max_depth=3, random_state=42)
tree_model.fit(X_train, y_train)

plt.figure(figsize=(18, 7))
plot_tree(tree_model, feature_names=features, impurity=False, rounded=True, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestRegressor

forest_model = RandomForestRegressor(n_estimators=100, random_state=42)
forest_model.fit(X_train, y_train)

forest_test_predictions = forest_model.predict(X_test)
print('Forest RMSE:', root_mean_squared_error(y_test, forest_test_predictions))
print('Forest R²:', r2_score(y_test, forest_test_predictions))

In [ ]:
importance = pd.Series(forest_model.feature_importances_, index=features)
importance.sort_values().plot.barh(figsize=(6, 3))
plt.xlabel('Feature importance in this fitted forest')
plt.tight_layout()
plt.show()

## 4. Tuning

In [ ]:
from sklearn.model_selection import KFold, cross_val_score, GridSearchCV

cv = KFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
forest_params = {'max_depth': [2, 4, 8, None]}

forest_search = GridSearchCV(
    RandomForestRegressor(n_estimators=100, random_state=42), forest_params,
    cv=cv, scoring='neg_root_mean_squared_error'
)
forest_search.fit(X_train, y_train)
forest_search.best_params_

In [ ]:
print('Best cross-validated RMSE:', -forest_search.best_score_)

Random folds vs spatial folds (whole 10 km blocks held out).

In [ ]:
from sklearn.model_selection import GroupKFold

best_forest = forest_search.best_estimator_
blocks_train = study.loc[X_train.index, 'block']

random_scores = -cross_val_score(best_forest, X_train, y_train, cv=cv,
                                 scoring='neg_root_mean_squared_error')
spatial_scores = -cross_val_score(best_forest, X_train, y_train, cv=GroupKFold(n_splits=5),
                                  groups=blocks_train, scoring='neg_root_mean_squared_error')
print('Random folds RMSE: ', random_scores.mean())
print('Spatial folds RMSE:', spatial_scores.mean())

## 5. Final test

In [ ]:
final_model = best_forest
pred_test = final_model.predict(X_test)

print('Final RMSE:', root_mean_squared_error(y_test, pred_test))
print('Final R²:', r2_score(y_test, pred_test))

In [ ]:
plt.scatter(y_test, pred_test, s=10, alpha=0.5)
plt.plot([0, y.max()], [0, y.max()], color='black', linestyle='--')
plt.xlabel('Observed (log rate)')
plt.ylabel('Predicted (log rate)')
plt.title('Random forest: test squares')
plt.show()

In [ ]:
tree_explainer = shap.TreeExplainer(final_model)
forest_shap = tree_explainer(X_test)
shap.plots.beeswarm(forest_shap)

## 6. Errors on the map
Residual = observed − predicted.

In [ ]:
test_map = study.loc[X_test.index].copy()
test_map['residual'] = y_test - pred_test

ax = study.plot(color='lightgrey', figsize=(7, 7))
test_map.plot(ax=ax, column='residual', cmap='RdBu', vmin=-1.5, vmax=1.5, legend=True)
plt.title('Random forest residuals (test squares)')
plt.axis('off')
plt.show()

## 7. Save results

In [ ]:
results = pd.DataFrame({
    'Model': ['Linear regression', 'Random forest'],
    'Test RMSE': [root_mean_squared_error(y_test, linear_test_predictions),
                  root_mean_squared_error(y_test, pred_test)],
    'Test R²': [r2_score(y_test, linear_test_predictions), r2_score(y_test, pred_test)],
    'SLAI coefficient': [coefficients['slai'], None],
    'Mean |SHAP| of SLAI': [np.abs(shap_values[:, 'slai'].values).mean(),
                            np.abs(forest_shap[:, 'slai'].values).mean()],
})
results.to_csv(f'../outputs/tables/results_{target}.csv', index=False)
results

Now change `target` to `'comparison_rate'` and run again.